In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import sklearn
import lightgbm

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams.update(
    {
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "axes.titlesize": 16,
        "axes.labelsize": 16,
        "pdf.fonttype": 42,
        "font.family": "Avenir",
        "font.size": 16,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.size": 5,
        "ytick.major.size": 5,
        "xtick.major.pad": 6.5,
        "ytick.major.pad": 6.5,
    }
)

In [ ]:
data = pd.read_csv("processed_data/dataInterpolated.csv")
data.columns

In [ ]:
use_data = data[['Unnamed: 0', 'VPCC_RSAM', 'VPPC_RSAM', 'VPNC_RSAM', 'VPRS_RSAM',
                 'VPNC_Intensity', 'CO2_ppm','lake_size', 'Eruption_Activity']]
display(use_data)

In [ ]:
dates = use_data['Unnamed: 0'].to_numpy()

In [ ]:
dates[100][:-6]

In [ ]:
dates = [datetime.strptime(i[:-6],'%Y-%m-%d %H:%M:%S') for i in dates]
print(dates[:5])
print((dates[1]-dates[0]).total_seconds())

In [ ]:
times = np.zeros_like(dates)[:-1]
for i in range(len(dates) - 1):
    times[i] = (dates[i+1]-dates[i]).total_seconds()
print(np.min(times), np.max(times)) #so all values are evenly spaced by a minute

In [ ]:
#making features and labels
full_data_arr = use_data.to_numpy()
full_data_arr.shape

In [ ]:
#evaluate on this many (minutes)
m = 720
#predicting eruption in next minutes
n = 720*4

#making array of starts to featurize next m minutes and label with n mins after that
size = full_data_arr.shape[0]
high_start = size - n - m #highest starting time for slice
starts = np.arange(high_start)[::m]
starts

In [ ]:
#slicing dataset
X = np.zeros((len(starts),5040))
Y = np.zeros((len(starts)))
for i in range(len(starts)):
    #features
    X[i,:] = full_data_arr[starts[i]:starts[i]+m, 1:-1].reshape(-1)
    erup = full_data_arr[starts[i]+m:starts[i]+m+n,-1]
    if np.max(erup)>0:
        Y[i] = 1


In [ ]:
print("data shape: ", X.shape)
print("label shape: ", Y.shape)

In [ ]:
#splitting up data 60% train, 30% val, 10% test

X_train, X_temp, Y_train, Y_temp = sklearn.model_selection.train_test_split(X,Y,test_size = 0.4, stratify = Y)
X_val, X_test, Y_val, Y_test = sklearn.model_selection.train_test_split(X_temp,Y_temp,test_size = 0.25, stratify = Y_temp)

In [ ]:
#tree types: 'dart', 'gbdt', 'rf'

In [ ]:
#training LightGBM
lgbm = lightgbm.LGBMClassifier(boosting_type = 'dart', learning_rate=0.1, n_estimators=200)
lgbm.fit(X_train, Y_train, eval_set=(X_val, Y_val))

In [ ]:
lgbm.score(X_val, Y_val)

In [ ]:
lgbm.score(X_test, Y_test)

In [ ]:
#getting ROC 
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax = ax.flatten()

y_probs = lgbm.predict_proba(X_test)[:,1]
fpr, tpr, thresholds = sklearn.metrics.roc_curve(Y_test, y_probs)
sklearn.metrics.RocCurveDisplay(fpr = fpr, tpr = tpr).plot(ax[0], color='#3779D5')
ax[0].set_box_aspect(1)
ax[0].text(
    0.02, 0.98, "A. ROC Curve",
    transform=ax[0].transAxes,

    fontsize=14,
    fontweight="bold",
    va="top",
    ha="left",
)

imp = lgbm.feature_importances_
folded = [0 for i in range(7)]
for i in range(len(imp)):
    folded[i%7] += imp[i]

labels = ax[1].plot(folded, 'o-', color='#3779D5')
ax[1].set_xticks(ticks = np.arange(7), labels = ['VPCC_RSAM', 'VPPC_RSAM', 'VPNC_RSAM', 'VPRS_RSAM',
       'VPNC_Intensity', 'CO2_ppm','lake_size'], rotation = 30, ha='right')
ax[1].set_box_aspect(1) 
ax[1].set_ylabel("Importance")
ax[1].set_xlabel("Feature")
ax[1].text(
    0.02, 0.98, "B. Relative Feature Importance, folded over time",
    transform=ax[1].transAxes,
    fontsize=14,
    fontweight="bold",
    va="top",
    ha="left",
)
plt.tight_layout()
plt.savefig('trees.pdf')

In [ ]:
imp = lgbm.feature_importances_
folded = [0 for i in range(720)]
for i in range(len(folded)):
    folded[i] = np.sum(imp[7*i:7*(i+1)])

plt.title("importance summed over 15 features per time")
plt.ylabel("importance")
plt.xlabel("minutes in inference window")
plt.plot(folded)

In [ ]:
#plotting val confusion matrix
sklearn.metrics.ConfusionMatrixDisplay(sklearn.metrics.confusion_matrix(Y_val, lgbm.predict(X_val)), display_labels=["No eruption", "Eruption"]).plot();